## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install torch-geometric
!pip install transformers sacremoses protobuf
!pip install peft  # LoRA
!pip install tqdm pandas
!pip install nltk rouge-score  # BLEU/ROUGE evaluation

## 2. Mount Google Drive & Set Data Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# IMPORTANT: Change these paths to match your Google Drive structure
DRIVE_DATA_DIR = '/content/drive/MyDrive/molecular_data/data/'

# Local paths (Colab's fast local disk)
LOCAL_DATA_DIR = '/content/data/'
CHECKPOINT_DIR = '/content/drive/MyDrive/molecular_data/checkpoints/'  # Save checkpoints to Drive

import shutil,os 

# Create local directory
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Copy data from Drive to local disk (MUCH faster for training!)
print("Copying data from Drive to local disk...")
print("   (This only happens once, training will be much faster)")

for filename in ['train_graphs.pkl', 'validation_graphs.pkl']:
    src = os.path.join(DRIVE_DATA_DIR, filename)
    dst = os.path.join(LOCAL_DATA_DIR, filename)
    
    if os.path.exists(src):
        if not os.path.exists(dst):
            print(f"   Copying {filename}...")
            shutil.copy(src, dst)
            print(f"   {filename} copied!")
        else:
            print(f"   {filename} already exists locally")
    else:
        print(f"   {filename} not found in Drive!")

# Data file paths (use LOCAL paths for training)
TRAIN_GRAPHS_PATH = LOCAL_DATA_DIR + 'train_graphs.pkl'
VAL_GRAPHS_PATH = LOCAL_DATA_DIR + 'validation_graphs.pkl'

print(f"\nLocal data directory: {LOCAL_DATA_DIR}")
print(f"Checkpoint directory (Drive): {CHECKPOINT_DIR}")

## 3. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast

from torch_geometric.data import Batch
from torch_geometric.utils import to_dense_batch
from torch_geometric.nn import GINEConv

from transformers import T5ForConditionalGeneration, T5Tokenizer, AutoConfig, AutoTokenizer, get_linear_schedule_with_warmup

from tqdm import tqdm
import pickle
import random
import os
from typing import Dict, List, Any
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

import gc

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. Data Utilities (from data_utils.py)

In [ ]:
# =========================================================
# Feature maps for atom and bond attributes
# =========================================================

x_map: Dict[str, List[Any]] = {
    'atomic_num': list(range(0, 119)),
    'chirality': [
        'CHI_UNSPECIFIED','CHI_TETRAHEDRAL_CW','CHI_TETRAHEDRAL_CCW','CHI_OTHER',
        'CHI_TETRAHEDRAL','CHI_ALLENE','CHI_SQUAREPLANAR','CHI_TRIGONALBIPYRAMIDAL',
        'CHI_OCTAHEDRAL',
    ],
    'degree': list(range(0, 11)),
    'formal_charge': list(range(-5, 7)),
    'num_hs': list(range(0, 9)),
    'num_radical_electrons': list(range(0, 5)),
    'hybridization': [
        'UNSPECIFIED','S','SP','SP2','SP3','SP3D','SP3D2','OTHER',
    ],
    'is_aromatic': [False, True],
    'is_in_ring': [False, True],
}

e_map: Dict[str, List[Any]] = {
    'bond_type': [
        'UNSPECIFIED','SINGLE','DOUBLE','TRIPLE','QUADRUPLE','QUINTUPLE','HEXTUPLE',
        'ONEANDAHALF','TWOANDAHALF','THREEANDAHALF','FOURANDAHALF','FIVEANDAHALF',
        'AROMATIC','IONIC','HYDROGEN','THREECENTER','DATIVEONE','DATIVE','DATIVEL',
        'DATIVER','OTHER','ZERO',
    ],
    'stereo': [
        'STEREONONE','STEREOANY','STEREOZ','STEREOE','STEREOCIS','STEREOTRANS',
    ],
    'is_conjugated': [False, True],
}


class MoleculeGenDataset(Dataset):
    """
    Dataset for molecular graph generation task.
    Loads pre-saved PyG graph objects.
    __getitem__ returns:
        - graph: The PyG Data object.
        - text: The raw description string
    """
    def __init__(self, graph_path: str):
        print(f"Loading graphs from: {graph_path}")
        with open(graph_path, 'rb') as f:
            self.graphs = pickle.load(f)
            
        if len(self.graphs) > 0 and not hasattr(self.graphs[0], 'description'):
            print("Warning: Loaded graphs do not appear to have a 'description' attribute.")
            
        print(f"Loaded {len(self.graphs)} graphs for generation task.")

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        graph = self.graphs[idx]
        text = getattr(graph, 'description', "No description available.")
        return graph, text


class BioT5Collate:
    """
    Custom collate class for the DataLoader with BioT5.
    Handles batching PyG graphs and tokenizing text separately for T5.
    For T5, we only need labels (target text), no need to concatenate with graph tokens.
    """
    def __init__(self, tokenizer, max_len=128):
        if isinstance(tokenizer, str):
            print(f"Initializing Tokenizer: {tokenizer}")
            self.tokenizer = AutoTokenizer.from_pretrained(tokenizer)
        else:
            self.tokenizer = tokenizer
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_len = max_len

    def __call__(self, batch):
        graphs = [item[0] for item in batch]
        raw_texts = [item[1] for item in batch]
        
        batched_graphs = Batch.from_data_list(graphs)
        
        # Tokenize target text (labels for T5 decoder)
        labels = self.tokenizer(
            raw_texts,
            padding=True,
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        return batched_graphs, labels['input_ids'], labels['attention_mask']

print("✅ Data utilities loaded!")

## 5. GNN Encoder Components

In [ ]:
class AtomEncoder(nn.Module):
    """
    Encoder for atom features in molecular graphs.
    Embeds atomic number, chirality, degree, formal charge,
    number of hydrogens, radical electrons, hybridization,
    aromaticity, and ring membership.
    
    Args:
        hidden_dim: Hidden dimension for embeddings
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.emb_atomic_num = nn.Embedding(119, hidden_dim)
        self.emb_chirality = nn.Embedding(9, hidden_dim)
        self.emb_degree = nn.Embedding(11, hidden_dim)
        self.emb_charge = nn.Embedding(12, hidden_dim)
        self.emb_num_hs = nn.Embedding(9, hidden_dim)
        self.emb_rad_el = nn.Embedding(5, hidden_dim)
        self.emb_hybrid = nn.Embedding(8, hidden_dim)
        self.emb_aromatic = nn.Embedding(2, hidden_dim)
        self.emb_ring = nn.Embedding(2, hidden_dim)
    
    def forward(self, x):
        x = x.long()
        atomic_num = self.emb_atomic_num(x[:, 0].clamp(0, 118))
        chirality = self.emb_chirality(x[:, 1].clamp(0, 8))
        degree = self.emb_degree(x[:, 2].clamp(0, 10))
        charge = self.emb_charge(x[:, 3].clamp(0, 11))
        num_hs = self.emb_num_hs(x[:, 4].clamp(0, 8))
        rad_el = self.emb_rad_el(x[:, 5].clamp(0, 4))
        hybrid = self.emb_hybrid(x[:, 6].clamp(0, 7))
        aromatic = self.emb_aromatic(x[:, 7].clamp(0, 1))
        ring = self.emb_ring(x[:, 8].clamp(0, 1))
        
        h = (atomic_num + chirality + degree + charge +
             num_hs + rad_el + hybrid + aromatic + ring)
        return h


class EdgeEncoder(nn.Module):
    """
    Encoder for edge/bond features in molecular graphs.
    Embeds bond type, stereo configuration, and conjugation status.
    
    Args:
        hidden_dim: Hidden dimension for embeddings
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.emb_bond_type = nn.Embedding(23, hidden_dim)
        self.emb_stereo = nn.Embedding(6, hidden_dim)
        self.emb_conjugated = nn.Embedding(2, hidden_dim)
    
    def forward(self, edge_attr):
        return self.emb_bond_type(edge_attr[:, 0]) + \
               self.emb_stereo(edge_attr[:, 1]) + \
               self.emb_conjugated(edge_attr[:, 2])


class MolGNN(nn.Module):
    """
    Graph Neural Network for molecular graph encoding.
    Uses GINEConv layers with residual connections to encode
    molecular structure into node embeddings.
    
    Args:
        hidden: Hidden dimension size
        out_dim: Output dimension (should match LLM embedding size)
        layers: Number of GNN layers
        dropout: Dropout rate
    """
    def __init__(self, hidden=256, out_dim=1024, layers=5, dropout=0.1):
        super().__init__()
        self.dropout = dropout
        self.atom_encoder = AtomEncoder(hidden)
        self.edge_encoder = EdgeEncoder(hidden)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(layers):
            mlp = nn.Sequential(
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            )
            self.convs.append(GINEConv(mlp, train_eps=True))
            self.norms.append(nn.LayerNorm(hidden))

        self.proj = nn.Sequential(
            nn.Linear(hidden, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, out_dim),
            nn.LayerNorm(out_dim)  
        )

    def forward(self, batch: Batch):
        h = self.atom_encoder(batch.x)
        edge_attr = self.edge_encoder(batch.edge_attr)
        
        for conv, norm in zip(self.convs, self.norms):
            h_in = h 
            h = conv(h, batch.edge_index, edge_attr=edge_attr)
            h = norm(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training) 
            h = h + h_in  # Residual connection 
        
        g = self.proj(h)
        return g, batch.batch



## 6. GraphBioT5 Model

In [ ]:
class GraphBioT5(nn.Module):
    """
    GraphBioT5: GNN + BioT5 for molecular graph captioning.
    
    The GNN extracts atom-level features which are projected to BioT5's
    encoder dimension, then fed as inputs_embeds to the T5 encoder.
    The T5 decoder generates the molecular description.
    
    Key improvement: Uses a text prompt prefix to help T5 understand the task.
    """
    def __init__(self, gnn_encoder, tokenizer, model_name='QizhiPei/biot5-base', 
                 freeze_gnn=False, use_lora=True, lora_r=8, lora_alpha=32, lora_dropout=0.1):
        super().__init__()
        self.gnn = gnn_encoder
        self.use_lora = use_lora
        self.model_name = model_name
        self.tokenizer = tokenizer
        
        # Task prompt - matches the target text format "The molecule is..."
        # This helps the model align encoder input with decoder output format
        self.prompt_text = "The molecule is "
        
        # GNN freezing option
        if freeze_gnn:
            print("Freezing GNN parameters...")
            for param in self.gnn.parameters():
                param.requires_grad = False
        else:
            print("GNN parameters are trainable")
        
        # Load BioT5
        print(f" Loading {model_name}...")
        self.biot5 = T5ForConditionalGeneration.from_pretrained(model_name)
        
        # Get dimensions
        self.biot5_dim = self.biot5.config.d_model  # Usually 768 for base
        # Get GNN output dimension from LayerNorm's normalized_shape (last layer is LayerNorm)
        gnn_out_dim = gnn_encoder.proj[-1].normalized_shape[0]
        
        print(f"BioT5 Hidden Dim: {self.biot5_dim}")
        print(f"GNN Output Dim: {gnn_out_dim}")
        
        # Projector: Map GNN output to BioT5 dimension
        self.projector = nn.Sequential(
            nn.Linear(gnn_out_dim, self.biot5_dim),
            nn.LayerNorm(self.biot5_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.biot5_dim, self.biot5_dim),
            nn.LayerNorm(self.biot5_dim)
        )
        
        # Apply LoRA 
        if use_lora:
            print(f"Applying LoRA (r={lora_r}, alpha={lora_alpha})")
            lora_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                bias="none",
                task_type=TaskType.SEQ_2_SEQ_LM,  
                target_modules=["q", "v", "k", "o", "wi", "wo"],  # T5 attention modules
            )
            self.biot5 = get_peft_model(self.biot5, lora_config)
            self.biot5.print_trainable_parameters()
        else:
            print("Full BioT5 fine-tuning (no LoRA)")
    
    def get_prompt_embeds(self, batch_size, device):
        """Get prompt embeddings to prepend to graph embeddings."""
        prompt_ids = self.tokenizer(
            self.prompt_text, 
            return_tensors="pt", 
            add_special_tokens=False
        ).input_ids.to(device)
        
        if self.use_lora:
            embed_tokens = self.biot5.base_model.model.encoder.embed_tokens
        else:
            embed_tokens = self.biot5.encoder.embed_tokens
        
        prompt_embeds = embed_tokens(prompt_ids)  # [1, prompt_len, dim]
        prompt_embeds = prompt_embeds.expand(batch_size, -1, -1)  # [batch, prompt_len, dim]
        return prompt_embeds, prompt_ids.shape[1]

    def forward(self, batch, labels=None, label_attention_mask=None, pad_token_id=0):
        """
        Training forward pass with prompt prefix.
        
        Args:
            batch: PyG Batch object containing molecular graphs
            labels: Tokenized target text (input_ids from tokenizer)
            label_attention_mask: Attention mask for labels
            pad_token_id: Padding token ID from tokenizer (default 0 for T5)
            
        Returns:
            loss: Cross-entropy loss for generation
        """
        device = next(self.parameters()).device
        
        # 1. GNN extracts atom-level features
        node_embeds, batch_index = self.gnn(batch)
        
        # 2. Convert to dense batch: [Batch, Max_Atoms, gnn_out_dim]
        graph_embeds, graph_mask = to_dense_batch(node_embeds, batch_index)
        batch_size = graph_embeds.shape[0]
        
        # 3. Project to BioT5 dimension: [Batch, Max_Atoms, biot5_dim]
        graph_embeds = self.projector(graph_embeds)
        
        # 4. Get prompt embeddings and concatenate: [prompt, graph]
        prompt_embeds, prompt_len = self.get_prompt_embeds(batch_size, device)
        inputs_embeds = torch.cat([prompt_embeds, graph_embeds], dim=1)
        
        # 5. Create attention mask for [prompt, graph]
        prompt_mask = torch.ones((batch_size, prompt_len), dtype=torch.long, device=device)
        full_attention_mask = torch.cat([prompt_mask, graph_mask.long()], dim=1)
        
        # 6. Prepare labels for T5
        # T5 expects labels with -100 for padding tokens
        if labels is not None:
            labels = labels.clone()
            labels[labels == pad_token_id] = -100  # Replace pad tokens with -100
        
        # 7. Forward through BioT5
        outputs = self.biot5(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attention_mask,
            labels=labels
        )
        
        return outputs.loss
    
    def generate(self, batch, tokenizer, max_new_tokens=100, num_beams=5):
        """
        Generate descriptions for molecular graphs with prompt prefix.
        
        Args:
            batch: PyG Batch object containing molecular graphs
            tokenizer: T5 tokenizer
            max_new_tokens: Maximum tokens to generate
            num_beams: Beam search width
            
        Returns:
            generated_ids: Tensor of generated token IDs
        """
        self.biot5.eval()
        device = next(self.parameters()).device
        
        with torch.no_grad():
            # 1. GNN extracts atom-level features
            node_embeds, batch_index = self.gnn(batch)
            
            # 2. Convert to dense batch
            graph_embeds, graph_mask = to_dense_batch(node_embeds, batch_index)
            batch_size = graph_embeds.shape[0]
            
            # 3. Project to BioT5 dimension
            graph_embeds = self.projector(graph_embeds)
            
            # 4. Get prompt embeddings and concatenate: [prompt, graph]
            prompt_embeds, prompt_len = self.get_prompt_embeds(batch_size, device)
            inputs_embeds = torch.cat([prompt_embeds, graph_embeds], dim=1)
            
            # 5. Create attention mask for [prompt, graph]
            prompt_mask = torch.ones((batch_size, prompt_len), dtype=torch.long, device=device)
            full_attention_mask = torch.cat([prompt_mask, graph_mask.long()], dim=1)
            
            # 6. Generate using T5 (Pure Beam Search for stable results)
            generated_ids = self.biot5.generate(
                inputs_embeds=inputs_embeds,
                attention_mask=full_attention_mask,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                do_sample=False,  # Pure beam search (no sampling)
                early_stopping=True,
                length_penalty=1.0,
                no_repeat_ngram_size=3,  # Avoid repetition
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
            
            return generated_ids
    
    def save_lora_weights(self, save_path):
        """Save only LoRA weights."""
        if self.use_lora:
            self.biot5.save_pretrained(save_path)
            print(f"LoRA weights saved to: {save_path}")
        else:
            print("LoRA not enabled, use torch.save() instead")
    
    def load_lora_weights(self, load_path):
        """Load LoRA weights."""
        if self.use_lora:
            self.biot5.load_adapter(load_path, adapter_name="default")
            print(f"LoRA weights loaded from: {load_path}")


print("GraphBioT5 loaded!")

## 7. Training & Evaluation Functions

In [ ]:
def print_generation_samples(model, val_loader, tokenizer, device, num_samples=2):
    """
    Print Ground Truth vs Generated text samples.
    
    Args:
        model: Trained model
        val_loader: Validation dataloader
        tokenizer: Tokenizer for decoding
        device: Device to run on
        num_samples: Number of samples to print
    """
    model.eval()
    
    batch = next(iter(val_loader))
    graph_batch, input_ids, attention_mask = batch
    graph_batch = graph_batch.to(device)
    
    print("\n--- Generation Check ---")
    
    with torch.no_grad():
        generated_ids = model.generate(
            graph_batch, 
            tokenizer, 
            max_new_tokens=50, 
            num_beams=3
        )
    
    indices = random.sample(range(len(generated_ids)), min(num_samples, len(generated_ids)))
    
    for idx in indices:
        generated_text = tokenizer.decode(generated_ids[idx], skip_special_tokens=True).strip()
        pred_text = model.prompt_text + generated_text
        true_text = tokenizer.decode(input_ids[idx], skip_special_tokens=True)
        
        print(f"[Sample {idx}]")
        print(f"   Real: {true_text[:100]}...")
        print(f"   Pred: {pred_text}")
    print("---------------------------\n")


def evaluate_loss(model, val_loader, device, tokenizer):
    """
    Compute average validation loss.
    
    Args:
        model: Model to evaluate
        val_loader: Validation dataloader
        device: Device to run on
        tokenizer: Tokenizer (for consistency with API)
        
    Returns:
        Average validation loss
    """
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in val_loader:
            graph_batch = batch[0].to(device)
            labels = batch[1].to(device)
            label_attention_mask = batch[2].to(device)
            
            with autocast(device_type='cuda'):
                loss = model(graph_batch, labels=labels, label_attention_mask=label_attention_mask,
                           pad_token_id=tokenizer.pad_token_id)
            
            total_loss += loss.item()
            
    avg_loss = total_loss / len(val_loader)
    return avg_loss


def save_checkpoint(model, optimizer, scaler, epoch, best_val_loss, save_path):
    """
    Save complete training state for checkpoint/resume functionality.
    
    Args:
        model: The model to save
        optimizer: Optimizer state
        scaler: GradScaler state for mixed precision
        epoch: Current epoch number
        best_val_loss: Best validation loss achieved so far
        save_path: Path to save the checkpoint file
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_val_loss': best_val_loss,
    }
    torch.save(checkpoint, save_path)
    print(f"Checkpoint saved: epoch {epoch+1}")


def load_checkpoint(model, optimizer, scaler, checkpoint_path, device):
    """
    Load training state for checkpoint/resume functionality.
    
    Args:
        model: The model to load state into
        optimizer: Optimizer to restore state
        scaler: GradScaler to restore state
        checkpoint_path: Path to the checkpoint file
        device: Device to map the checkpoint to
        
    Returns:
        Tuple of (start_epoch, best_val_loss)
    """
    if not os.path.exists(checkpoint_path):
        print("No checkpoint found, starting from scratch")
        return 0, float('inf')
    
    print(f"Loading checkpoint from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    
    print(f"Resumed from epoch {checkpoint['epoch']+1}")
    print(f"   Best val loss so far: {best_val_loss:.4f}")
    
    return start_epoch, best_val_loss


def train_model(model, train_loader, val_loader, tokenizer, epochs=15, lr=5e-5, 
                save_dir='checkpoints', resume=False, grad_accum_steps=1,
                gnn_lr=1e-4, projector_lr=1e-4, warmup_epochs=1):
    """
    Train the GraphBioT5 model with checkpoint/resume, gradient accumulation,
    layer-wise learning rates, and warmup scheduler.
    
    Args:
        model: GraphBioT5 model to train
        train_loader: Training data loader
        val_loader: Validation data loader
        tokenizer: Tokenizer for decoding during evaluation
        epochs: Total number of training epochs
        lr: Learning rate for BioT5 (lower)
        save_dir: Directory to save checkpoints
        resume: Whether to resume training from checkpoint
        grad_accum_steps: Number of gradient accumulation steps (simulates larger batch)
        gnn_lr: Learning rate for GNN (higher)
        projector_lr: Learning rate for Projector (higher)
        warmup_epochs: Number of warmup epochs
        
    Returns:
        Trained model
    """
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    
    model = model.to(device)
    
    # Layer-wise learning rates: GNN/Projector need higher LR (randomly initialized)
    # BioT5 needs lower LR (already pretrained)
    optimizer = torch.optim.AdamW([
        {'params': model.gnn.parameters(), 'lr': gnn_lr},
        {'params': model.projector.parameters(), 'lr': projector_lr},
        {'params': model.biot5.parameters(), 'lr': lr},
    ])
    
    print(f"Layer-wise Learning Rates:")
    print(f"   GNN:       {gnn_lr}")
    print(f"   Projector: {projector_lr}")
    print(f"   BioT5:     {lr}")
    
    scaler = GradScaler()
    
    # Warmup scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = len(train_loader) * warmup_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    print(f"Warmup: {warmup_epochs} epoch(s) ({warmup_steps} steps)")
    
    # Checkpoint paths
    checkpoint_path = os.path.join(save_dir, "biot5_training_checkpoint.pt")
    best_model_path = os.path.join(save_dir, "biot5_best_model.pth")
    
    # Resume from checkpoint if requested
    start_epoch = 0
    best_val_loss = float('inf')
    
    if resume:
        start_epoch, best_val_loss = load_checkpoint(
            model, optimizer, scaler, checkpoint_path, device
        )
    
    if start_epoch >= epochs:
        print(f"Training already completed ({epochs} epochs)")
        return model

    print(f"Gradient Accumulation: {grad_accum_steps} steps")
    print(f"Effective batch size: {train_loader.batch_size * grad_accum_steps}")

    for epoch in range(start_epoch, epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # --- Training Step ---
        model.train()
        train_loss = 0
        optimizer.zero_grad()  # Zero gradients at epoch start
        
        loop = tqdm(train_loader, desc="Train")
        
        for step, batch in enumerate(loop):
            graph_batch = batch[0].to(device)
            labels = batch[1].to(device)
            label_attention_mask = batch[2].to(device)
            
            loss = model(graph_batch, labels=labels, label_attention_mask=label_attention_mask,
                           pad_token_id=tokenizer.pad_token_id)
            loss = loss / grad_accum_steps  # Scale loss for accumulation
            
            scaler.scale(loss).backward()
            
            # Update parameters every grad_accum_steps
            if (step + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()  # Update learning rate
                optimizer.zero_grad()
            
            train_loss += loss.item() * grad_accum_steps
            loop.set_postfix(loss=loss.item() * grad_accum_steps, lr=scheduler.get_last_lr()[0])
            
            # Periodically clear GPU cache
            if step % 100 == 0:
                torch.cuda.empty_cache()
            
        avg_train = train_loss / len(train_loader)
        print(f"Train Loss: {avg_train:.4f}")
        
        # --- Evaluation Step ---
        torch.cuda.empty_cache()  # Clear cache before evaluation
        
        if val_loader:
            avg_val = evaluate_loss(model, val_loader, device, tokenizer)
            print(f"Val Loss:   {avg_val:.4f}")
            
            # Generation samples
            print_generation_samples(model, val_loader, tokenizer, device)

            # Save best model
            if avg_val < best_val_loss:
                best_val_loss = avg_val
                torch.save(model.state_dict(), best_model_path)
                print("Best Model Updated!")
        
        # Save checkpoint after each epoch
        save_checkpoint(model, optimizer, scaler, epoch, best_val_loss, checkpoint_path)
        torch.save(model.state_dict(), os.path.join(save_dir, "biot5_last_model.pth"))
    
    print("\nTraining completed!")
    return model



print("Training functions loaded!")

## 8. Configuration & Hyperparameters

In [ ]:
# Memory optimization settings
BATCH_SIZE = 2          
GRAD_ACCUM_STEPS = 8     
LEARNING_RATE = 5e-5     
EPOCHS = 4               
MAX_TEXT_LEN = 128         

# Model settings - BioT5
MODEL_NAME = "QizhiPei/biot5-base"
GNN_HIDDEN_DIM = 256     
GNN_LAYERS = 5            

# LoRA settings
USE_LORA = True
LORA_R = 16              
LORA_ALPHA = 32          
LORA_DROPOUT = 0.05

# Freeze GNN to save memory
FREEZE_GNN = False

print("Configuration (Memory Optimized for BioT5):")
print(f"   Batch Size: {BATCH_SIZE} (x{GRAD_ACCUM_STEPS} accum = effective {BATCH_SIZE*GRAD_ACCUM_STEPS})")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Model: {MODEL_NAME}")
print(f"   Max Text Length: {MAX_TEXT_LEN}")
print(f"   GNN Hidden Dim: {GNN_HIDDEN_DIM}")
print(f"   GNN Layers: {GNN_LAYERS}")
print(f"   Freeze GNN: {FREEZE_GNN}")
print(f"\nLoRA Settings:")
print(f"   Use LoRA: {USE_LORA}")
print(f"   LoRA Rank (r): {LORA_R}")
print(f"   LoRA Alpha: {LORA_ALPHA}")

## 9. Initialize Model & Data

In [ ]:
# 1. Get BioT5 configuration
print(f"Checking configuration for {MODEL_NAME}...")
llm_config = AutoConfig.from_pretrained(MODEL_NAME)

# 2. Detect model dimension (T5 uses d_model)
if hasattr(llm_config, 'd_model'):
    target_dim = llm_config.d_model
else:
    raise ValueError("Could not detect model dimension automatically!")
    
print(f"✅ Detected BioT5 Dimension: {target_dim}")

# 3. Initialize tokenizer (T5Tokenizer for BioT5)
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4. Data loading with BioT5Collate
collate_fn = BioT5Collate(tokenizer, max_len=MAX_TEXT_LEN)

print("Loading Data...")
train_dataset = MoleculeGenDataset(TRAIN_GRAPHS_PATH)
val_dataset = MoleculeGenDataset(VAL_GRAPHS_PATH)

# Note: num_workers=0 for Colab to avoid multiprocessing issues
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn, 
    num_workers=0
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn, 
    num_workers=0
)

print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Val batches: {len(val_loader)}")

In [ ]:
# Clear GPU memory first
import gc
gc.collect()
torch.cuda.empty_cache()

# 5. Initialize GNN
gnn = MolGNN(
    hidden=GNN_HIDDEN_DIM, 
    layers=GNN_LAYERS, 
    out_dim=target_dim  # Output matches BioT5 d_model
)

# 6. Initialize GraphBioT5 with LoRA
model = GraphBioT5(
    gnn,
    tokenizer=tokenizer,  # Pass tokenizer for prompt embedding
    model_name=MODEL_NAME,
    freeze_gnn=FREEZE_GNN,     # Freeze GNN to save memory
    use_lora=USE_LORA,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nParameter Summary:")
print(f"   Total:     {total_params:,}")
print(f"   Trainable: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"   Frozen:    {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")

## 10. Start Training!

In [ ]:


# Clear GPU memory before training
import gc
gc.collect()
torch.cuda.empty_cache()


os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Set to False for first run, True to resume after crash
RESUME_TRAINING = False

# Layer-wise learning rate settings
GNN_LR = 1e-4         # Higher LR for randomly initialized GNN
PROJECTOR_LR = 1e-4   # Higher LR for randomly initialized Projector
WARMUP_EPOCHS = 1     # Warmup epochs

trained_model = train_model(
    model, 
    train_loader, 
    val_loader, 
    tokenizer=tokenizer,
    epochs=EPOCHS, 
    lr=LEARNING_RATE,
    save_dir=CHECKPOINT_DIR,
    resume=RESUME_TRAINING,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    gnn_lr=GNN_LR,
    projector_lr=PROJECTOR_LR,
    warmup_epochs=WARMUP_EPOCHS
)

print("\nTraining Complete!")
print(f"Models saved to: {CHECKPOINT_DIR}")

## Load Best Model

In [ ]:
# Load the best model for inference
best_model_path = os.path.join(CHECKPOINT_DIR, "biot5_best_model.pth")

if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path))
    print(f"✅ Loaded best model from: {best_model_path}")
else:
    print("Best model not found, using current model")

## Test Generation

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

# Get a batch from validation set
batch = next(iter(val_loader))
graph_batch, input_ids, attention_mask = batch
graph_batch = graph_batch.to(device)

print("Testing Generation...\n")

with torch.no_grad():
    generated_ids = model.generate(
        graph_batch, 
        tokenizer, 
        max_new_tokens=100, 
        num_beams=5
    )

# Show first 5 samples
for i in range(min(5, len(generated_ids))):
    generated_text = tokenizer.decode(generated_ids[i], skip_special_tokens=True).strip()
    pred_text = model.prompt_text + generated_text
    true_text = tokenizer.decode(input_ids[i], skip_special_tokens=True)
    
    print(f"---------- Sample {i+1} ----------")
    print(f"\nGround Truth:")

    print(f"   {true_text}")    
    print()

    print(f"Generated:")    
    print(f"   {pred_text}")

## Generate Submission CSV



In [ ]:
TEST_GRAPHS_PATH = LOCAL_DATA_DIR + 'test_graphs.pkl'

OUTPUT_CSV_PATH = CHECKPOINT_DIR + 'BioT5_submission.csv'

test_src = os.path.join(DRIVE_DATA_DIR, 'test_graphs.pkl')
test_dst = TEST_GRAPHS_PATH
if os.path.exists(test_src) and not os.path.exists(test_dst):
    print("Copying test_graphs.pkl to local disk...")
    shutil.copy(test_src, test_dst)
    print("Done!")


class TestMoleculeDataset(Dataset):
    """
    Dataset for test set (no descriptions, only graphs)
    """
    def __init__(self, graph_path: str):
        print(f"Loading test graphs from: {graph_path}")
        with open(graph_path, 'rb') as f:
            self.graphs = pickle.load(f)
        self.ids = [g.id for g in self.graphs]
        print(f"Loaded {len(self.graphs)} test graphs.")

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        return self.graphs[idx]


def test_collate_fn(batch):
    """
    Collate function for test set (graphs only, no text labels).
    
    Args:
        batch: List of graph Data objects

    Returns:
        Batched graph object
    """
    return Batch.from_data_list(batch)

print("✅ Test dataset utilities ready!")

In [ ]:
@torch.no_grad()
def generate_kaggle_submission(model, test_path, tokenizer, device, output_csv, batch_size=8, max_new_tokens=150):
    """
    Generate descriptions for test set and save to CSV for Kaggle submission.
    
    Args:
        model: Trained GraphBioT5 model
        test_path: Path to test_graphs.pkl
        tokenizer: BioT5 tokenizer
        device: Device to run on
        output_csv: Path to save output CSV
        batch_size: Batch size for generation (smaller = less memory)
        max_new_tokens: Maximum tokens to generate
    """
    model.eval()
    model = model.to(device)
    
    # Load test dataset
    test_dataset = TestMoleculeDataset(test_path)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=test_collate_fn
    )
    
    print(f"Generating descriptions for {len(test_dataset)} test molecules...")
    print(f"   Batch size: {batch_size}")
    print(f"   Max new tokens: {max_new_tokens}")
    
    results = []
    
    for batch_idx, graph_batch in enumerate(tqdm(test_loader, desc="Generating")):
        graph_batch = graph_batch.to(device)
        
        # Get IDs for this batch
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(test_dataset))
        batch_ids = test_dataset.ids[start_idx:end_idx]
        
        # Generate descriptions
        generated_ids = model.generate(
            graph_batch,
            tokenizer,
            max_new_tokens=max_new_tokens,
            num_beams=5
        )
        
        # Decode and save
        for i, test_id in enumerate(batch_ids):
            if i < len(generated_ids):
                generated_text = tokenizer.decode(generated_ids[i], skip_special_tokens=True).strip()
                description = model.prompt_text + generated_text
                
                results.append({
                    'ID': test_id,
                    'description': description
                })
        
        # Print progress every 50 batches
        if batch_idx % 50 == 0 and batch_idx > 0:
            print(f"   Generated {len(results)}/{len(test_dataset)} descriptions...")
    
    # Create DataFrame and save
    results_df = pd.DataFrame(results)
    
    # Sort by ID to ensure correct order
    results_df['ID'] = results_df['ID'].astype(int)
    results_df = results_df.sort_values('ID').reset_index(drop=True)
    
    # Save to CSV
    results_df.to_csv(output_csv, index=False)
    
    print(f"\n{'='*60}")
    print(f"Kaggle submission file generated!")
    print(f"Saved to: {output_csv}")
    print(f"Total samples: {len(results_df)}")
    print(f"{'='*60}")
    
    # Show first few examples
    print("\nFirst 5 generated descriptions:")
    for i in range(min(5, len(results_df))):
        row = results_df.iloc[i]
        print(f"\nID {row['ID']}:")
        print(f"   {row['description'][:200]}...")
    
    return results_df

print("✅ Generation function ready!")


In [ ]:


best_model_path = os.path.join(CHECKPOINT_DIR, "biot5_best_model.pth")
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path))
    print(f"Loaded best model from: {best_model_path}")
else:
    print("Using current model (biot5_best_model.pth not found)")

# Generate submission
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

submission_df = generate_kaggle_submission(
    model=model,
    test_path=TEST_GRAPHS_PATH,
    tokenizer=tokenizer,
    device=device,
    output_csv=OUTPUT_CSV_PATH,
    batch_size=8,        # Reduce if OOM error
    max_new_tokens=150   # Adjust based on expected description length
)

print(f"\nDownload your submission from: {OUTPUT_CSV_PATH}")

In [ ]:
# Read and display submission info
submission = pd.read_csv(OUTPUT_CSV_PATH)

print("Submission File Summary:")
print(f"   Shape: {submission.shape}")
print(f"   Columns: {list(submission.columns)}")
print(f"   ID range: {submission['ID'].min()} - {submission['ID'].max()}")
print(f"\nSample entries:")
print(submission.head(10))

# Check for any issues
print(f"\nData Quality Check:")
print(f"   Missing descriptions: {submission['description'].isna().sum()}")
print(f"   Empty descriptions: {(submission['description'] == '').sum()}")
print(f"   Unique descriptions: {submission['description'].nunique()}")

# Download link for Colab
from google.colab import files
files.download(OUTPUT_CSV_PATH)
print(f"\nFile downloaded! Ready to submit to Kaggle.")